In [ ]:
%matplotlib inline


In [ ]:
# מטרות הפרויקט:
# לנתח את התנהגות הלקוחות תוך פילוח לפי מאפיינים דמוגרפיים שונים כמו סטטוס משפחתי, קבוצות הכנסה, גיל, והרכב המשפחה (כמות ילדים).
# לזהות אילו קטגוריות מוצרים (יין, בשר, דגים, מתוקים, מוצרי זהב ופירות) מובילות לרווחיות הגבוהה ביותר בקרב קבוצות לקוחות שונות,
#ניתוח קמפיינים ע"י פילוג של התנהגות צרכנית שנקבעת ע"י היחשפות&היענות לקמפיין
# לבחון כיצד פילוחי הלקוחות השונינם (כמות ילדים בית, קבוצת השתכרות, קבוצת גיל ורמת צרכנות) משפיעים על התנהגות הלקוחות מבחינת סכומי הרכישה והערוצים בהם הם בוחרים לרכוש (חנות, אתר, קטלוג או מבצעים).
# להציע לעסק המלצות עסקיות מדויקות ומבוססות נתונים, לשיפור האסטרטגיה השיווקית והגדלת ההכנסות.

# נושא הניתוח והשאלות המחקריות בפרויקט:
# נושא הניתוח: ניתוח ופילוח הרגלי צריכה של לקוחות בעסק מוצרי מזון ומשקאות, תוך השוואה בין קבוצות דמוגרפיות ותגובות לקמפיינים שיווקיים.

# שאלות המחקר המרכזיות:
# 1. כיצד המצב המשפחתי (רווק, נשוי, גרוש וכו') משפיע על ההכנסות, הרגלי הקנייה (יין, בשר, דגים, מתוקים, זהב) ההכנסה הממוצעת ללקוח והסטיית תקן ?
# 2. כיצד החשיפה והתגובה לקמפיינים שיווקיים (קמפיין דיגיטלי/לא דיגיטלי או אי-חשיפה) משפיעות על סך ההכנסות, על כמות הרכישות בכל ערוץ (אתר, חנות, קטלוג, מבצעים), ועל גובה ההוצאה הממוצעת?
# 3. כיצד רמת ההכנסה של הלקוחות משפיעה על דפוסי הרכישה וההוצאה בכל קטגוריית מוצר?
# 4. האם קיים קשר בין גיל הלקוחות ורמת הרכישות הכללית שלהם לבין כמות ואופן השימוש בקטלוגים כאמצעי לרכישה?
# 5. כיצד משפיעה כמות הילדים בבית (ללא ילדים, ילד אחד, מספר ילדים) על ההכנסות, העדפות הרכישה (קטגוריות מוצרים), ובחירת ערוצי הקנייה השונים?

# הכלים והטכניקות בהם השתמשתי להצגת המסקנות:
# SQL:
# בניית שאילתות מתקדמות לפילוח וניתוח נתונים, כולל שימוש ב-CTE, חישוב אחוזים, ממוצעים וסטיות תקן לניתוח סטטיסטי מעמיק של ההתנהגות הצרכנית.

# Python ו-Jupyter Notebook:
# שימוש ב-Pandas לחיבור למסד הנתונים, איחוד נתונים, ניתוחים סטטיסטיים מפורטים וחישובי אחוזים.
# שימוש נרחב ב-Matplotlib וב-Seaborn ליצירת גרפים ברורים ואסתטיים להצגת מגמות, כמו תרשימי עמודות מוערמים, גרפי Scatter, תרשימי פאי, Heatmap, ותרשימי Violin.
# חישובי מדדים כמו ממוצע, סטיית תקן, אחוזים מתוך סך הרכישות, והתפלגות רכישות לפי ערוצי מכירה וקבוצות לקוחות שונות.

# סיכום כללי של גישת העבודה:
# ניתוח הנתונים נעשה בצורה שיטתית, כאשר כל חלק במחקר מתמקד בשאלות ספציפיות ומציג תוצאות בצורה חזותית וסטטיסטית,  
# מה שמאפשר הבנה עמוקה וברורה של דפוסי הצריכה והעדפות הלקוחות. 
# ניתוחים אלה מאפשרים קבלת החלטות מבוססות דאטה לשיפור הביצועים העסקיים של החברה.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
from sqlalchemy import create_engine
from IPython.display import display, Markdown

# Set plot style
sns.set_style("whitegrid")

# Database connection
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# Load data
queries = {
    'customers': "SELECT CustomerID, Education, Marital_Status AS MaritalStatus, Income, CustomerJoinDate, Gender, Kidhome, Teenhome, Age, Year_Birth FROM Customers",
    'activity': "SELECT CustomerID, Recency, TotalDealsPurchases, TotalWebPurchases, TotalCatalogPurchases, TotalStorePurchases, MonthlyWebVisits FROM Activity",
    'purchases': "SELECT CustomerID, MntWines, MntFruits, MntMeatProducts, MntFishProducts, MntSweetProducts, MntGoldProducts FROM Purchases",
    'campaign_accepted': "SELECT CustomerID, Response, Complain, AcceptedCmp1, AcceptedCmp2, AcceptedCmp3, AcceptedCmp4, AcceptedCmp5 FROM Campaign_Accepted"
}

tables = {name: pd.read_sql(query, engine) for name, query in queries.items()}

# Merge data
df = (tables['customers']
      .merge(tables['activity'], on='CustomerID')
      .merge(tables['purchases'], on='CustomerID')
      .merge(tables['campaign_accepted'], on='CustomerID'))

# Data preprocessing
df['CustomerJoinDate'] = pd.to_datetime(df['CustomerJoinDate'])
df['TotalSpending'] = df[['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProducts']].sum(axis=1)
df['TotalTransactions'] = df[['TotalWebPurchases', 'TotalStorePurchases', 'TotalCatalogPurchases', 'TotalDealsPurchases']].sum(axis=1)
df['AvgSpending'] = df['TotalSpending'] / df['TotalTransactions'].replace(0, pd.NA)  # Avoid division by zero
df['AcceptedCampaigns'] = df[['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']].sum(axis=1)
df['AgeCategory'] = pd.cut(df['Age'], bins=[18, 30, 45, 60, 69, 80], labels=["29-39", "39-49", "49-59", "59-69", "69-79"], include_lowest=True)
df['IncomeCategory'] = pd.cut(df['Income'], bins=[0, 25000, 55000, 150000], labels=["Low", "Medium", "High"])

# Calculate KPIs
kpis = {
    'Avg Spending Per Customer': int(df['TotalSpending'].sum() / df.shape[0]),
    'Total Customers': df['CustomerID'].nunique(),
    'Total Revenue': int(df['TotalSpending'].sum()),
    'Avg Transactions Per Customer': round(df['TotalTransactions'].mean(), 1)
}

# Data overview
def display_data_overview():
    display(Markdown("### 🔍 סקירה כללית של הנתונים"))
    print(f"מבנה הנתונים המלא (df.info):\n{df.info()}\n")
    
    for name, table in tables.items():
        print(f"📌 ראש הטבלה - {name}:")
        display(table.head())
        print(f"📊 סטטיסטיקות תיאוריות - {name}:")
        display(table.describe().round(2))
        print(f"🔍 ערכים חסרים - {name}:")
        missing = table.isnull().sum()
        print(missing[missing > 0] if missing.sum() > 0 else "אין ערכים חסרים\n")

    display(Markdown("### ✅ מדדי מפתח (KPIs)"))
    for key, value in kpis.items():
        print(f"{key}: {value}")

# Column descriptions
def display_column_descriptions():
    display(Markdown("### 📌 תיאור משמעות העמודות העיקריות"))
    descriptions = {
        "Customers": [
            ("CustomerID", "מזהה לקוח ייחודי"), ("Year_Birth", "שנת לידה"), ("Education", "רמת השכלה"),
            ("MaritalStatus", "סטטוס משפחתי"), ("Income", "הכנסה שנתית"), ("CustomerJoinDate", "תאריך הצטרפות"),
            ("Gender", "מגדר"), ("Kidhome", "מספר ילדים"), ("Teenhome", "מספר מתבגרים"), ("Age", "גיל")
        ],
        "Purchases": [
            ("MntWines", "סכום רכישות יין"), ("MntMeatProducts", "סך רכישות בשר"), ("MntFishProducts", "סך רכישות דגים"),
            ("MntSweetProducts", "רכישות מוצרים מתוקים"), ("MntGoldProducts", "רכישות מוצרי זהב"), ("MntFruits", "רכישות פירות")
        ],
        "Campaign Accepted": [
            ("AcceptedCmp1-5", "קמפיינים שהלקוח נענה להם (1/0)"), ("Response", "תגובה לקמפיין האחרון"), 
            ("Complain", "האם הלקוח התלונן")
        ],
        "Activity": [
            ("Recency", "ימים מאז הרכישה האחרונה"), ("TotalWebPurchases", "רכישות באתר"), 
            ("TotalCatalogPurchases", "רכישות מקטלוג"), ("TotalStorePurchases", "רכישות בחנות"), 
            ("TotalDealsPurchases", "רכישות במבצע"), ("MonthlyWebVisits", "ביקורים חודשיים באתר")
        ]
    }
    for category, cols in descriptions.items():
        print(f"\n{category}:")
        for col, desc in cols:
            print(f"{col} – {desc}")

# Visualizations
def create_visualizations():
    # Treemap of product categories
    product_categories = {'MntWines': 'Wine', 'MntMeatProducts': 'Meat', 'MntGoldProducts': 'Gold',
                         'MntSweetProducts': 'Sweet', 'MntFishProducts': 'Fish', 'MntFruits': 'Fruit'}
    category_sum = df[list(product_categories.keys())].sum()
    labels = [f"{product_categories[col]}\n{val/1000:.2f}K" for col, val in category_sum.items()]
    
    plt.figure(figsize=(12, 6))
    squarify.plot(sizes=category_sum.values, label=labels, alpha=.85, color=sns.color_palette("coolwarm", len(category_sum)))
    plt.title("Total Revenue by Product Category", fontsize=16, weight='bold')
    plt.axis('off')
    plt.show()

    print("💡 יין הוא המוצר הנמכר ביותר, ולאחריו בשר. מוצרים כמו זהב ומתוקים נמכרים פחות.")


    # Average income by age category
    income_by_age = df.groupby("AgeCategory", observed=False)["Income"].mean().round().reset_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=income_by_age, x="AgeCategory", y="Income", palette="muted")
    for i, row in income_by_age.iterrows():
        plt.text(i, row.Income + 1000 if pd.notna(row.Income) else 1000, 
                f"{int(row.Income/1000)}K" if pd.notna(row.Income) else "No Data", 
                ha='center', fontsize=11, fontweight='bold', color='black' if pd.notna(row.Income) else 'gray')
    plt.title("Average Income by Age Category", fontsize=16, fontweight="bold")
    plt.ylabel("Avg Income")
    plt.xlabel("Age Category")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()


    print("💡 מרבית הלקוחות נמצאים בטווח גילאים 49-59. המשכורת של הלקוחות היא ביחס ישיר עם הגיל,\nהלקוחות שבני 69-79 הם הבזבזנים ביותר (פר לקוח) ורוכשים גם הכי הרבה רכישות (פר לקוח),\nאף על פי שהם אינם מכניסים הכי הרבה כסף לחברה.")

    
    # Purchase channel percentages
    channel_sums = df[['TotalWebPurchases', 'TotalStorePurchases', 'TotalCatalogPurchases', 'TotalDealsPurchases']].sum()
    channel_percentages = (channel_sums / channel_sums.sum() * 100).round(2)
    channel_percentages = channel_percentages.rename({'TotalWebPurchases': 'Web', 'TotalStorePurchases': 'Store', 
                                                    'TotalCatalogPurchases': 'Catalog', 'TotalDealsPurchases': 'Deals'})
    
    plt.figure(figsize=(6, 8))
    bottom = 0
    colors = ['#34495E', '#F1C40F', '#D35400', '#A93226']
    for i, (channel, percent) in enumerate(channel_percentages.items()):
        plt.bar('Channels', percent, bottom=bottom, label=f"{channel} ({percent}%)", color=colors[i])
        bottom += percent
    plt.title("Total Purchase % by Channel", fontsize=16, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('off')
    plt.show()
    print("💡 רכישות בחנויות פיזיות בשיעור הגבוה ביותר")

    # Histogram of total spending
    plt.figure(figsize=(12, 6))
    sns.histplot(df['TotalSpending'], bins=40, kde=False, color='#2980B9', edgecolor='black')
    plt.title("Distribution of Customer Total Spending", fontsize=16, fontweight='bold')
    plt.xlabel("Total Spending")
    plt.ylabel("Count")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
    print("💡קבוצת הרכישה הנמוכה מבין הלקוחות אחראית לרוב הרכישות, כאשר הטווח העיקרי עומד על עד 1300 שקל")


# Execute analysis
display_data_overview()
display_column_descriptions()
create_visualizations()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define product categories for spending
spending_cols = ["MntWines", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProducts", "MntFruits"]

# Calculate total spending per customer
purchases["TotalSpending"] = purchases[spending_cols].sum(axis=1)

# Merge customer and purchase data
df = customers.merge(purchases, on="CustomerID")

# Calculate key metrics by marital status
marital_stats = df.groupby("Marital_Status").agg(
    CustomerCount=("CustomerID", "count"),
    AvgIncome=("Income", "mean"),
    TotalRevenue=("TotalSpending", "sum"),
    AvgSpendingPerCustomer=("TotalSpending", "mean"),
    SpendingStdDev=("TotalSpending", "std"),
    **{col: (col, "sum") for col in spending_cols}
)

# Calculate percentages for each product category
for col in spending_cols:
    marital_stats[f"{col}_Percentage"] = (marital_stats[col] / marital_stats["TotalRevenue"] * 100).round(1)

# Sort by total revenue
marital_stats = marital_stats.sort_values("TotalRevenue", ascending=False)

# Define product category labels and colors to match the chart
product_labels = ["Fish", "Fruit", "Gold", "Meat", "Sweet", "Wine"]
colors = ["#1F77B4", "#FFBB78", "#8C564B", "#D62728", "#C49C94", "#2CA02C"]

# 📊 Chart 1: Stacked Bar Chart - Spending by Product Category and Marital Status
plt.figure(figsize=(10, 6))
percentages = marital_stats[[f"{col}_Percentage" for col in spending_cols]]

# Plot stacked bars
bottom = pd.Series(0, index=percentages.index)
for i, col in enumerate(spending_cols):
    plt.barh(percentages.index, percentages[f"{col}_Percentage"], left=bottom, color=colors[i], label=product_labels[i])
    for j, value in enumerate(percentages[f"{col}_Percentage"]):
        if value > 0:  # Only annotate if the value is non-zero
            plt.text(bottom[j] + value/2, j, f"{value}%", ha='center', va='center', fontsize=10, color='white', weight='bold')
    bottom += percentages[f"{col}_Percentage"]

# Customize the chart
plt.title("Spending by Marital Status and Product Category", fontsize=14, fontweight="bold")
plt.xlabel("")
plt.ylabel("Marital Status", fontsize=12)
plt.legend(title="Categories", loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=6, fontsize=10)
plt.gca().invert_yaxis()  # Invert y-axis to match the chart
plt.grid(False)


plt.tight_layout()
plt.show()

# Insights for Chart 1
print("💡 תובנות לגרף 1:")
print("יין ובשר הם המוצרים הפופולריים ביותר בכל קבוצות הסטטוס המשפחתי.")
print("נשואים בולטים באהבתם לשילוב של השניים. לאחריהם, תכשיטים מובילים ברוב הקבוצות, מלבד אלמנים, שמעדיפים דגים.")
print("זוגות צורכים הכי הרבה יין, רווקים – הכי הרבה פירות, וגרושים מובילים ברכישת תכשיטים.")
fig, ax1 = plt.subplots(figsize=(10, 6))

# ערכים בסיסיים
avg_spending = marital_stats["AvgSpendingPerCustomer"]
std_spending = marital_stats["SpendingStdDev"]
statuses = marital_stats.index
bar_colors = ["#1F77B4", "#FFBB78", "#8C564B", "#2CA02C", "#C49C94"]

# 🟦 גרף עמודות – ממוצע הוצאה
ax1.bar(range(len(statuses)), avg_spending, color=bar_colors, alpha=0.7, label="Average Spending")
ax1.set_ylabel("Avg Spending Amount", fontsize=12)
ax1.set_xticks(range(len(statuses)))
ax1.set_xticklabels(statuses)
ax1.set_ylim(0, avg_spending.max() + 100)

# טקסט מעל כל עמודה
for i, avg in enumerate(avg_spending):
    ax1.text(i, avg + 15, f"{int(avg)}", ha='center', fontsize=10, fontweight='bold')

# 🔴 גרף קו – סטיית תקן
ax2 = ax1.twinx()
ax2.plot(range(len(statuses)), std_spending, color='red', marker='o', markersize=10, linewidth=3, label="Spending Std Dev")
ax2.set_ylabel("Spending Std Dev", fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, std_spending.max() + 150)  # ✅ מרווח גדול יותר למעלה

# טקסט מעל הנקודות האדומות
for i, std in enumerate(std_spending):
    ax2.text(i, std + 20, f"{int(std)}", ha='center', fontsize=10, fontweight='bold', color='red')

# 🧠 טייטל ו־אגדה
plt.title("Average Spending and Standard Deviation by Marital Status", fontsize=16, fontweight="bold")
fig.legend(loc="upper right", bbox_to_anchor=(1, 1), bbox_transform=ax1.transAxes)

plt.grid(axis='y', linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()



# Insights for Chart 2
print("💡 תובנות לגרף 2:")
print("קבוצת הנשואים היא היציבה והמובילה מבחינת היקף צריכה – גם מבחינת הוצאה ממוצעת גבוהה, וגם במספר הלקוחות.")
print("לעומתם, הרווקים והאלמנים מתאפיינים בפיזור הוצאות נמוך ובהוצאה כוללת נמוכה – מה שמצביע על פוטנציאל לשיפור דרך מיקוד שיווקי, במיוחד רווקים שהם הקבוצה השלישית בגודלה.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# Database connection
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# Load tables
customers = pd.read_sql("SELECT * FROM Customers", engine)
campaign_accepted = pd.read_sql("SELECT * FROM Campaign_Accepted", engine)
purchases = pd.read_sql("SELECT * FROM Purchases", engine)
activity = pd.read_sql("SELECT * FROM Activity", engine)

# Updated customer categorization function
def categorize_customer(row):
    exposed = (
        row["AcceptedCmp1"] == 1 or
        row["AcceptedCmp2"] == 1 or
        row["AcceptedCmp3"] == 1 or
        row["AcceptedCmp4"] == 1 or
        row["AcceptedCmp5"] == 1
    )
    responded = row["Response"] == 1

    if exposed and responded:
        return "Exposed & Responded"
    elif exposed and not responded:
        return "Exposed & Not Responded"
    else:
        return "Not Exposed"

# Add the column to the campaign table
campaign_accepted["CustomerCategory"] = campaign_accepted.apply(categorize_customer, axis=1)

# Merge data
df = (customers
      .merge(campaign_accepted, on="CustomerID", how="left")
      .merge(purchases, on="CustomerID", how="left")
      .merge(activity, on="CustomerID", how="left"))

# Calculate total revenue
df["TotalRevenue"] = (
    df["MntWines"] + 
    df["MntMeatProducts"] + 
    df["MntFishProducts"] + 
    df["MntSweetProducts"] + 
    df["MntFruits"] + 
    df["MntGoldProducts"]
)

# Group by customer category
df_grouped = df.groupby("CustomerCategory").agg(
    CustomerCount=("CustomerID", "count"),
    TotalRevenue=("TotalRevenue", "sum"),
    AvgSpendingPerCustomer=("TotalRevenue", "mean"),
    SpendingStdDev=("TotalRevenue", "std"),
    TotalPurchases=("TotalDealsPurchases", "sum"),
    DiscountPurchases=("TotalDealsPurchases", "sum"),
    WebPurchases=("TotalWebPurchases", "sum"),
    CatalogPurchases=("TotalCatalogPurchases", "sum"),
    StorePurchases=("TotalStorePurchases", "sum"),
).reset_index()

# Calculate purchase channel percentages
df_grouped["TotalPurchases"] = (
    df_grouped["DiscountPurchases"] +
    df_grouped["WebPurchases"] +
    df_grouped["CatalogPurchases"] +
    df_grouped["StorePurchases"]
)

df_grouped["DiscountPurchasesPercentage"] = (df_grouped["DiscountPurchases"] / df_grouped["TotalPurchases"] * 100).round(1)
df_grouped["WebPurchasesPercentage"] = (df_grouped["WebPurchases"] / df_grouped["TotalPurchases"] * 100).round(1)
df_grouped["CatalogPurchasesPercentage"] = (df_grouped["CatalogPurchases"] / df_grouped["TotalPurchases"] * 100).round(1)
df_grouped["StorePurchasesPercentage"] = (df_grouped["StorePurchases"] / df_grouped["TotalPurchases"] * 100).round(1)


# 📊 Chart 1: Donut Chart - Customer Distribution by Segment
plt.figure(figsize=(10, 8))

# Colors to match the Power BI image
colors = ["#50C878", "#4A90E2", "#F28C38"]

# Labels with customer count (without percentages)
labels = [f"{category}\n({count})" for category, count in zip(df_grouped["CustomerCategory"], df_grouped["CustomerCount"])]

# Create the pie chart (donut style) without autopct
wedges, texts = plt.pie(
    df_grouped["CustomerCount"], 
    labels=labels, 
    startangle=140,
    colors=colors, 
    explode=(0.05, 0.05, 0.05), 
    shadow=False,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, 
    textprops={'fontsize': 14, 'weight': 'bold', 'color': 'black'}
)

# Add a circle in the center to make it a donut chart
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

# Title in English
plt.title("Customer Distribution by Segment", fontsize=16, fontweight="bold", pad=20)

# Display the chart
plt.show()

# Insights for Chart 1
print("💡 תובנות לגרף 1:")
print("מרבית הלקוחות (כ-73%) לא נחשפו כלל לקמפיינים ולא הגיבו, בעוד שרק אחוז קטן נחשף והגיב.")
print("מבחינת הכנסות, שתי הקבוצות שלא נחשפו לקמפיינים יורדים בשיעורם משמעותית (כ 56% מההכנסות) ושתי הקבוצות שכן נחשפות לקמפיינים מגדילים ביותר מפי 2 את שיעורם בדונאט ההכנסות.")

# 📊 Final Chart: Avg Spending (Left Y) & Std Dev (Right Y) with Separate Scales
sns.set_style("whitegrid")

# 🎨 צבעים מודרניים
bar_colors = ["#4A90E2", "#F28C38", "#50C878"]
line_color = "#8E44AD"  # סגול מודרני
left_y_color = "black"
right_y_color = "black"

fig, ax1 = plt.subplots(figsize=(10, 6))

# 🟦 ממוצע הוצאות
bars = ax1.bar(
    df_grouped["CustomerCategory"],
    df_grouped["AvgSpendingPerCustomer"],
    color=bar_colors,
    alpha=0.85,
    label="AVG Spending Per Customer"
)

# 🏷️ טקסט על העמודות
for bar, value in zip(bars, df_grouped["AvgSpendingPerCustomer"]):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        value + 40,
        f"{int(value):,}",
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold',
        color="black"
    )

# ⬅️ גבולות ציר שמאל
ax1.set_ylim(0, 1600)

# ➕ סטיית תקן - ציר ימין
ax2 = ax1.twinx()
ax2.set_ylim(0, 1400)
ax2.set_yticks(np.arange(0, 1401, 200))

# 📈 קו step לסטיית תקן
ax2.plot(
    df_grouped["CustomerCategory"],
    df_grouped["SpendingStdDev"],
    color=line_color,
    marker='s',
    markersize=8,
    linewidth=3,
    linestyle='--',
    drawstyle='steps-mid',
    label="Spending Std Dev"
)

# 🏷️ טקסט לסטיית תקן
for i, value in enumerate(df_grouped["SpendingStdDev"]):
    ax2.annotate(
        f"{int(value)}",
        (i, value + 40),
        fontsize=12,
        fontweight='bold',
        color=line_color,
        ha='center',
        va='bottom'
    )

# 🧭 צירי Y
ax1.set_ylabel("AVG Spending Per Customer (₪)", fontsize=12, fontweight='bold', color=left_y_color)
ax2.set_ylabel("Spending Std Dev (₪)", fontsize=12, fontweight='bold', color=right_y_color)
ax1.tick_params(axis='y', colors=left_y_color)
ax2.tick_params(axis='y', colors=right_y_color)

# 🧭 ציר X
ax1.set_xlabel("Customer Category", fontsize=12, fontweight='bold')
ax1.set_xticks(range(len(df_grouped)))
ax1.set_xticklabels(df_grouped["CustomerCategory"], rotation=0, fontsize=10, fontweight='bold')

# 🎯 כותרת
plt.title("AVG Spending Per Customer & Spending Std Dev", fontsize=14, fontweight='bold', pad=20)

# 📏 רשת
ax1.grid(axis='y', linestyle="--", alpha=0.7)

# 🧾 מקרא
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=10, frameon=True)

plt.tight_layout()
plt.show()


# Insights for Chart 2
print("💡 תובנות לגרף 2:")
print("האנשים שלא חשופים לקמפיינים, הם הלקוחות האורגניים, הם יחפשו בטבעיות יותר הנחות ומבצעים כי הם לא מודעים לקמפיינים.")
print("הלקוחות הממומנים (אלה שחשופים לקמפיינים) מבזבזים בממוצע פי 2.5 יותר מהלקוחות האורגניים.")
print("ברמת סטיית התקן: המדדים נראים יותר מאוזנים אך עדיין התובנה מתחזקת שקמפיינים ממומנים מביאים רכישות שמנות יותר.")

# 📊 Chart 3: Stacked Bar Chart - Purchase Distribution by Channel
df_purchase_channels = df_grouped[[
    "CustomerCategory",
    "DiscountPurchasesPercentage",
    "WebPurchasesPercentage",
    "CatalogPurchasesPercentage",
    "StorePurchasesPercentage"
]].copy()

fig, ax = plt.subplots(figsize=(10, 6))

# Colors to match the Power BI image
colors = ["#4A90E2", "#F28C38", "#50C878", "#D62728"]

# Plot stacked bar chart
df_purchase_channels.set_index("CustomerCategory").plot(
    kind="bar", 
    stacked=True, 
    color=colors, 
    width=0.8, 
    ax=ax
)

# Customize labels and title
plt.xlabel("Customer Category", fontsize=12, fontweight='bold')
plt.ylabel("Purchase Distribution (%)", fontsize=12, fontweight='bold')
plt.title("Purchase Distribution by Channel", fontsize=14, fontweight='bold', pad=20)

# Customize x and y axes
plt.xticks(rotation=0, fontsize=10, fontweight='bold')
plt.yticks(np.arange(0, 101, 20), fontsize=10, fontweight='bold')
ax.set_ylim(0, 100)

# Add legend
plt.legend(
    title="", 
    fontsize=10,
    loc="upper center", 
    bbox_to_anchor=(0.5, 1.05), 
    ncol=4, 
    frameon=False,
    labels=["Discount Purchase %", "Store Purchases %", "Catalog Purchases %", "Web Purchases %"]
)

# Add percentage labels inside bars
for container in ax.containers:
    ax.bar_label(
        container, 
        fmt='%.1f%%', 
        label_type='center', 
        fontsize=10, 
        fontweight='bold', 
        color="white"
    )

# Add grid
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.xaxis.grid(False)

plt.show()

# Insights for Chart 3
print("💡 תובנות לגרף 3:")
print("לקוחות שנחשפו לקמפיינים מבצעים יותר רכישות דרך הקטלוג והאונליין, בעוד שאלו שלא נחשפו קונים יותר דרך הנחות וחנויות פיזיות.")
print("האנשים שלא חשופים לקמפיינים, הם הלקוחות האורגניים, הם יחפשו בטבעיות יותר הנחות ומבצעים כי הם לא מודעים לקמפיינים אלה שכן מודעים מחפשים פחות את ההנחה ומונעים באופן ממומן.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sqlalchemy import create_engine
from IPython.display import display

# 🔹 Database connection
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# 🔹 Load data from the database
query = """
    SELECT c.CustomerID, c.IncomeCategory, 
           p.MntWines, p.MntMeatProducts, p.MntFishProducts, 
           p.MntSweetProducts, p.MntFruits, p.MntGoldProducts
    FROM Customers c
    INNER JOIN Purchases p ON c.CustomerID = p.CustomerID
"""
df = pd.read_sql(query, engine)

# 🧮 Calculate total spending per customer
df["TotalSpendingPerCustomer"] = df[
    ["MntWines", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntFruits", "MntGoldProducts"]
].sum(axis=1)

# 📊 Summary table by income category
income_purchases = df.groupby("IncomeCategory").agg(
    CustomerCount=("CustomerID", "count"),
    TotalWineRevenue=("MntWines", "sum"),
    TotalMeatRevenue=("MntMeatProducts", "sum"),
    TotalFishRevenue=("MntFishProducts", "sum"),
    TotalSweetRevenue=("MntSweetProducts", "sum"),
    TotalFruitRevenue=("MntFruits", "sum"),
    TotalGoldRevenue=("MntGoldProducts", "sum"),
    TotalRevenue=("TotalSpendingPerCustomer", "sum"),
    AvgSpendingPerCustomer=("TotalSpendingPerCustomer", "mean"),
    SpendingStdDev=("TotalSpendingPerCustomer", "std"),
    MinSpending=("TotalSpendingPerCustomer", "min"),
    MaxSpending=("TotalSpendingPerCustomer", "max")
).reset_index()

# 🎯 Calculate percentages for each product
for product in ['Wine', 'Meat', 'Fish', 'Sweet', 'Fruit', 'Gold']:
    income_purchases[f'{product}Percentage'] = (
        income_purchases[f'Total{product}Revenue'] / income_purchases["TotalRevenue"] * 100
    ).round(2)

# 🧹 Sort by total revenue
income_purchases = income_purchases.sort_values(by="TotalRevenue", ascending=False)

# 📋 Display the table
display(income_purchases)

# 🔹 Calculate overall KPIs
total_revenue_all = income_purchases["TotalRevenue"].sum()

# 🔹 Calculate Income Category % for each category (relative to total revenue)
income_purchases["Income Category %"] = (income_purchases["TotalRevenue"] / total_revenue_all * 100).round(1)

# 🔹 Ensure correct order for Chart 1
desired_order = ["Very High", "High", "Middle", "Low"]
income_purchases_chart1 = income_purchases.set_index("IncomeCategory").loc[desired_order].reset_index()

# ----------------------------------------
# 📊 Chart 1: Bar Chart - Total Revenue by Income Group (Using Plotly Graph Objects for Tooltips)
# ----------------------------------------
# Colors to match Power BI
colors = {"Very High": "#1C4E80", "High": "#A5D8DD", "Middle": "#7E2E2E", "Low": "#D32D41"}

# Create the bar chart with Plotly Graph Objects
fig = go.Figure()

# Add bars for each category
for index, row in income_purchases_chart1.iterrows():
    fig.add_trace(
        go.Bar(
            x=[row["IncomeCategory"]],
            y=[row["TotalRevenue"]],
            name=row["IncomeCategory"],
            marker_color=colors[row["IncomeCategory"]],
            text=f"{row['TotalRevenue']/1e6:.2f}M",
            textposition='outside',
            textfont=dict(size=12, color="black", family="Arial, bold"),
            customdata=[[row["CustomerCount"], row["AvgSpendingPerCustomer"], row["SpendingStdDev"], row["Income Category %"]]],
            hovertemplate=(
                "<b>%{x}</b><br>" +
                "Total Revenue: %{y:,.0f}<br>" +
                "Total Customers: %{customdata[0]}<br>" +
                "AVG Spending Per Customer: %{customdata[1]:.1f}<br>" +
                "Spending StdDev: %{customdata[2]:.1f}<br>" +
                "Income Category %: %{customdata[3]}%<extra></extra>"
            )
        )
    )

# Update layout for better appearance
fig.update_layout(
    title=dict(
        text="Total Revenue by Income Group",  # Removed the income_category_text
        font=dict(size=14, family="Arial, bold"),
        x=0.5,
        xanchor="center"
    ),
    xaxis_title=dict(text="Income Category", font=dict(size=12, family="Arial, bold")),
    yaxis_title=dict(text="Total Revenue", font=dict(size=12, family="Arial, bold")),
    xaxis=dict(tickfont=dict(size=10, family="Arial, bold")),
    yaxis=dict(
        tickfont=dict(size=10, family="Arial"),
        gridcolor="lightgray",
        gridwidth=1,
        range=[0, 700000]  # Set Y-axis range to 700,000
    ),
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=400,
    width=600
)

# Display the chart
fig.show()

print("💡 תובנות לגרף 1:")
print("קבוצת ההכנסה הגבוהה והגבוהה מאוד מניעים את רובן המוחלט של ההכנסות, עם ממוצע רכישה גבוה כאשר בקבוצה הגבוהה מאוד יש אנשים שמפריזים עם כוח קנייתם. עם זאת שתי הקבוצות עם סטיית תקן יציבה ודומה")
print("קבוצת ההכנסה הבינונית מייצרים ממוצע סכום הנמוך משמעותית מהקבוצות הגבוהות, אך לפי סטיית התקן נראה כי הוא לא רחוק מהקבוצות הגבוהות, מה שמעיד על יכולת צריכה גבוהה ויציבה יחסית")
print("קבוצת ההכנסה הקטנה מכילים מעט לקוחות עם מדדים נמוכים יחסית לשאר הקבוצות")

# ----------------------------------------
# 📊 Chart 3: Stacked Bar Chart - Product Spending Distribution by Income Group
# ----------------------------------------
plt.figure(figsize=(10, 6))
product_cols = ['FishPercentage', 'FruitPercentage', 'GoldPercentage', 'MeatPercentage', 'SweetPercentage', 'WinePercentage']
product_labels = ['Fish', 'Fruit', 'Gold', 'Meat', 'Sweet', 'Wine']
colors = ["#1C4E80", "#A5D8DD", "#7E2E2E", "#D32D41", "#F4A261", "#2A9D8F"]  # Colors to match Power BI

ax = income_purchases.set_index("IncomeCategory")[product_cols].plot(
    kind='bar', 
    stacked=True, 
    color=colors, 
    width=0.8
)

# Add percentage labels inside bars
for container in ax.containers:
    ax.bar_label(
        container, 
        fmt='%.2f%%', 
        label_type='center', 
        fontsize=10, 
        color='white'
    )

plt.xlabel("Income Category", fontsize=12, fontweight='bold')
plt.ylabel("Percentage of Total Spending", fontsize=12, fontweight='bold')
plt.title("Product Spending Distribution by Income Group", fontsize=14, fontweight='bold', pad=50)  # Increased pad to make space for legend
plt.xticks(rotation=0, fontsize=10, fontweight='bold')
plt.yticks(np.arange(0, 101, 20), fontsize=10, fontweight='bold')
plt.grid(axis='y', linestyle="--", alpha=0.7)

# Adjust legend position to be in the white space below the title
plt.legend(
    title="", 
    labels=product_labels, 
    loc="upper center", 
    bbox_to_anchor=(0.5, 1.1),  # Position just below the title
    ncol=6, 
    frameon=False,
    fontsize=10
)

plt.tight_layout()
plt.show()

print("💡 תובנות לגרף 3:")
print("לקוחות בעלי הכנסה נמוכה מעדיפים תכשיטים במיוחד (צריכת ראווה)")
print("בנוסף, חוץ מיין ובשר, הם קונים באופן יחסי יותר מכולם בשאר הקטגוריות")
print("אפשר להבחין שככל שרמת ההכנסה של אותו לקוח גבוהה יותר ככה הוא צורך פחות יין ובשר ויותר את שאר הקטגוריות")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import display  # הצגת טבלה

# 🔹 התחברות למסד הנתונים
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# 🔹 טעינת הנתונים בפעם אחת עם JOIN במקום קריאות נפרדות
query = """
    SELECT c.CustomerID, c.Age, 
           a.TotalCatalogPurchases, a.TotalWebPurchases, a.TotalStorePurchases, a.TotalDealsPurchases
    FROM Customers c
    INNER JOIN Activity a ON c.CustomerID = a.CustomerID
"""
df = pd.read_sql(query, engine)

# 🔹 יצירת קבוצות גיל בהתאם לנתונים
bins = [29, 39, 49, 59, 69, 79]
labels = ["29-39", "39-49", "49-59", "59-69", "69-79"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

# 🔹 חישוב רכישות כוללות
df["TotalPurchases"] = df[["TotalCatalogPurchases", "TotalWebPurchases", "TotalStorePurchases", "TotalDealsPurchases"]].sum(axis=1)

# 🔹 קביעת רמות רכישה
def classify_purchase_level(total):
    if total <= 10:
        return 'Low Purchases (<= 10)'
    elif 11 <= total <= 20:
        return 'Medium Purchases (11-20)'
    else:
        return 'High Purchases (> 20)'
        

df["PurchaseLevel"] = df["TotalPurchases"].apply(classify_purchase_level)

# 🔹 יצירת טבלת סיכום
catalog_stats = df.groupby(["AgeGroup", "PurchaseLevel"], observed=False).agg(
    NumberOfCustomers=("CustomerID", "count"),
    TotalCatalogPurchases=("TotalCatalogPurchases", "sum"),
    AvgCatalogPurchasesPerCustomer=("TotalCatalogPurchases", "mean"),
    StdDevCatalogPurchases=("TotalCatalogPurchases", "std")
).reset_index()

# 🔹 חישוב יחס רכישות קטלוג מכלל הרכישות
total_purchases = df.groupby("AgeGroup", observed=False)["TotalCatalogPurchases"].sum()
catalog_stats["CatalogPurchaseRatio"] = catalog_stats.apply(
    lambda row: row["TotalCatalogPurchases"] / total_purchases[row["AgeGroup"]] if total_purchases[row["AgeGroup"]] > 0 else 0, axis=1
)

# Pivot להצגת סכומים לכל PurchaseLevel בקבוצת גיל
pivot_data = catalog_stats.pivot(index="AgeGroup", columns="PurchaseLevel", values="TotalCatalogPurchases")[["High Purchases (> 20)", "Medium Purchases (11-20)", "Low Purchases (<= 10)"]]

# ציור Stacked Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
pivot_data.plot(kind="bar", stacked=True, ax=ax, color=["#2C6B7A", "#C57B57", "#E4C66C"])

# הוספת תוויות
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + height / 2,
                f"{int(height)}",
                ha='center', va='center',
                fontsize=10, fontweight="bold", color="white"
            )

ax.set_title("Purchase Level Category", fontsize=16, weight="bold")
ax.set_ylabel("Sum of TotalCatalogPurchases", fontsize=12)
ax.set_xlabel("AgeCategory", fontsize=12)
ax.legend(title="Purchase Level")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

import plotly.graph_objects as go

# סידור הצירים
x_order = ["High Purchases (> 20)", "Medium Purchases (11-20)", "Low Purchases (<= 10)"]
y_order = ["29-39", "39-49", "49-59", "59-69", "69-79"]

# בניית מטריצות לערכים
z_matrix = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="AvgCatalog").reindex(index=y_order, columns=x_order).values
custom_matrix = merged.pivot(index="AgeGroup", columns="PurchaseLevel")  # נשתמש בהמשך לטולטיפ

# טקסט שיופיע בתוך כל תא
text_matrix = np.round(z_matrix, 2).astype(str)

# יצירת customdata: שלוש שכבות – avg, ratio, std
avg = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="AvgCatalog").reindex(index=y_order, columns=x_order).values
ratio = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="CatalogPurchaseRatio").reindex(index=y_order, columns=x_order).values
std = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="StdDevCatalogPurchases").reindex(index=y_order, columns=x_order).values

# הכנסת כל שלוש השכבות ל-customdata בצורה תקנית
customdata = np.stack([avg, ratio, std], axis=-1)

# 🎯 יצירת הגרף
fig = go.Figure(data=go.Heatmap(
    z=z_matrix,
    x=x_order,
    y=y_order,
    text=text_matrix,
    texttemplate="%{text}",
    colorscale='BrBG',
    colorbar=dict(title="Avg Purchases"),
    customdata=customdata,
    hovertemplate=
        "<b>Age Group:</b> %{y}<br>" +
        "<b>Purchase Level:</b> %{x}<br>" +
        "<b>Avg Catalog Purchases:</b> %{customdata[0]:.2f}<br>" +
        "<b>Catalog Purchase Ratio:</b> %{customdata[1]:.2f}<br>" +
        "<b>Std Dev (Catalog):</b> %{customdata[2]:.2f}<extra></extra>"
))

fig.update_layout(
    title="Avg Catalog Purchases per Customer by Age & Purchase Level",
    xaxis_title="Purchase Level",
    yaxis_title="Age Group",
    width=950,
    height=600,
    font=dict(family="Arial", size=12)
)

fig.show()





In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# התחברות לדאטהבייס
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# טעינת הטבלאות
customers = pd.read_sql("SELECT * FROM Customers", engine)
campaign_accepted = pd.read_sql("SELECT * FROM Campaign_Accepted", engine)
purchases = pd.read_sql("SELECT * FROM Purchases", engine)
activity = pd.read_sql("SELECT * FROM Activity", engine)


def categorize_customer(row):
    exposed = (row["AcceptedCmp1"] == 1) or (row["AcceptedCmp2"] == 1) or (row["AcceptedCmp3"] == 1) or (row["AcceptedCmp4"] == 1) or (row["AcceptedCmp5"] == 1)
    responded = row["Response"] == 1

    if exposed and responded:
        return "Exposed & Responded"
    elif exposed and not responded:
        return "Exposed & Not Responded"
    elif not exposed and not responded:
        return "Not Exposed & Not Responded"
    elif not exposed and responded:
        return "Not Exposed but Responded (Non-Digital)"

# Add the column to the campaign table
campaign_accepted["CustomerCategory"] = campaign_accepted.apply(categorize_customer, axis=1)




df = (customers
      .merge(campaign_accepted, on="CustomerID", how="left")
      .merge(purchases, on="CustomerID", how="left")
      .merge(activity, on="CustomerID", how="left"))

df["TotalRevenue"] = (
    df["MntWines"] + df["MntMeatProducts"] + df["MntFishProducts"] +
    df["MntSweetProducts"] + df["MntFruits"] + df["MntGoldProducts"]
)



df_grouped = df.groupby("CustomerCategory").agg(
    CustomerCount=("CustomerID", "count"),
    TotalRevenue=("TotalRevenue", "sum"),
    AvgSpendingPerCustomer=("TotalRevenue", "mean"),
    SpendingStdDev=("TotalRevenue", "std"),
    TotalPurchases=("TotalDealsPurchases", "sum"),
    DiscountPurchases=("TotalDealsPurchases", "sum"),
    WebPurchases=("TotalWebPurchases", "sum"),
    CatalogPurchases=("TotalCatalogPurchases", "sum"),
    StorePurchases=("TotalStorePurchases", "sum"),
).reset_index()

# חישוב אחוזי רכישה לפי ערוץ מכירה
df_grouped["TotalPurchases"] = (
    df_grouped["DiscountPurchases"] +
    df_grouped["WebPurchases"] +
    df_grouped["CatalogPurchases"] +
    df_grouped["StorePurchases"]
)

df_grouped["DiscountPurchasesPercentage"] = (df_grouped["DiscountPurchases"] / df_grouped["TotalPurchases"]) * 100
df_grouped["WebPurchasesPercentage"] = (df_grouped["WebPurchases"] / df_grouped["TotalPurchases"]) * 100
df_grouped["CatalogPurchasesPercentage"] = (df_grouped["CatalogPurchases"] / df_grouped["TotalPurchases"]) * 100
df_grouped["StorePurchasesPercentage"] = (df_grouped["StorePurchases"] / df_grouped["TotalPurchases"]) * 100



# השוואה לנתונים ב-SQL
print(f"סך ההכנסות: {df_grouped['TotalRevenue'].sum():,.2f} ₪")
print(f"ממוצע הוצאה ללקוח: {df_grouped['AvgSpendingPerCustomer'].mean():,.2f} ₪")
print(f"סטיית תקן של הוצאות: {df_grouped['SpendingStdDev'].mean():,.2f} ₪")


plt.figure(figsize=(10, 8))

# Colors
colors = sns.color_palette("pastel", len(df_grouped))

# Labels with customer count in English
labels = [f"{category}\n({count})" for category, count in zip(df_grouped["CustomerCategory"], df_grouped["CustomerCount"])]

# Creating the pie chart
plt.pie(df_grouped["CustomerCount"], labels=labels, autopct="%1.1f%%", startangle=140, 
        colors=colors, explode=(0.03, 0.03, 0.07, 0.12), shadow=False, 
        wedgeprops={'edgecolor': 'black', 'linewidth': 1.5}, textprops={'fontsize': 16, 'weight': 'bold'})

# Title in English
plt.title("Customer Distribution by Segment", fontsize=20, fontweight="bold", pad=20)

# Display the chart
plt.show()
print("רוב הלקוחות אינם מעורבים כאשר מדובר בקמפיינים שיווקים , לא כל מי שצופה בקמפיין נענה בדרך דיגיטלית.")





plt.figure(figsize=(10, 6))
sns.barplot(x="CustomerCategory", y="TotalRevenue", hue="CustomerCategory", data=df_grouped, palette="Blues_d", legend=False)

plt.xlabel("Customer Category", fontsize=14, fontweight='bold')
plt.ylabel("Total Revenue ($)", fontsize=14, fontweight='bold')
plt.title("Total Revenue by Customer Segment", fontsize=16, fontweight='bold')

plt.xticks(rotation=45, fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle="--", alpha=0.7)

# Display values above each bar
for index, value in enumerate(df_grouped["TotalRevenue"]):
    plt.text(index, value + 20000, f"${value:,.0f}", ha='center', fontsize=12, fontweight='bold', color="black")

plt.show()

print("אותם הלקוחות שלא חשופים לקמפיינים קונים במצטבר יותר מכל קבוצה אחרת .")

import matplotlib.pyplot as plt
import seaborn as sns

# 🔹 הגדרת עיצוב מודרני
sns.set_style("whitegrid")

# 🔹 צבעים מודרניים לכל ציר
bar_colors = sns.color_palette("Oranges", len(df_grouped))
line_color = "#355C7D"  # כחול עמוק
left_y_color = "#D35400"  # כתום כהה
right_y_color = "#355C7D"  # כחול כהה

# 🔹 יצירת גרף עמודות (הוצאה ממוצעת ללקוח)
fig, ax1 = plt.subplots(figsize=(10, 6))
bars = ax1.bar(df_grouped["CustomerCategory"], df_grouped["AvgSpendingPerCustomer"], 
               color=bar_colors, alpha=0.85, label="Avg Spending Per Customer")

# 🔹 הצמדת מספרים לקצה העליון של כל עמודה
for bar, value in zip(bars, df_grouped["AvgSpendingPerCustomer"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{value:,.1f}₪", 
             ha='center', va='bottom', fontsize=12, fontweight='bold', color="black",
             bbox=dict(facecolor="white", edgecolor="none", alpha=0.7))  # רקע לבן לקריאות

# 🔹 יצירת ציר Y נוסף עבור סטיית התקן
ax2 = ax1.twinx()
ax2.set_ylim(0, df_grouped["SpendingStdDev"].max() * 1.3)  # קנה מידה נכון לסטיית התקן

# 🔹 יצירת גרף קו של סטיית התקן
ax2.plot(df_grouped["CustomerCategory"], df_grouped["SpendingStdDev"], 
         color=line_color, marker='o', markersize=8, linewidth=2.5, linestyle="-", label="Spending Std Dev")

# 🔹 העלאת המספרים של קו סטיית התקן למעלה כדי לא להסתיר כלום
for i, txt in enumerate(df_grouped["SpendingStdDev"]):
    ax2.annotate(f"{txt:.1f}₪", (i, df_grouped["SpendingStdDev"][i] + 50),  # הרמה של 50 מעל לנקודה
                 fontsize=12, fontweight='bold', color=line_color, ha='center',
                 bbox=dict(facecolor="white", edgecolor="none", alpha=0.7))  # רקע לבן לקריאות

# 🔹 התאמת הצירים כך שייראו אחידים
ax1.set_ylim(0, df_grouped["AvgSpendingPerCustomer"].max() * 1.3)  # ציר ימין לממוצע
ax2.set_ylim(0, df_grouped["SpendingStdDev"].max() * 1.3)  # ציר שמאל לסטיית התקן

# 🔹 התאמת צבעי הצירים
ax1.yaxis.label.set_color(left_y_color)
ax2.yaxis.label.set_color(right_y_color)
ax1.tick_params(axis='y', colors=left_y_color)
ax2.tick_params(axis='y', colors=right_y_color)

# 🔹 התאמת צירים ושמות
ax1.set_xlabel("Customer Category", fontsize=14, fontweight='bold')
ax1.set_ylabel("Avg Spending Per Customer (₪)", fontsize=14, fontweight='bold', color=left_y_color)
ax2.set_ylabel("Spending Std Dev (₪)", fontsize=14, fontweight='bold', color=right_y_color)

# ✅ **תיקון האזהרה כאן:**
ax1.set_xticks(range(len(df_grouped)))  # קביעת מספר ה-Ticks
ax1.set_xticklabels(df_grouped["CustomerCategory"], rotation=25, ha='right', fontsize=12, fontweight='bold')

# 🔹 הוספת רשת גרפית
ax1.grid(axis='y', linestyle="--", alpha=0.7)

# 🔹 הוספת מקרא
ax2.legend(loc="upper right", fontsize=12, frameon=True)

# 🔹 כותרת מרכזית
plt.title("Avg Spending Per Customer & Standard Deviation by Segment", fontsize=16, fontweight='bold', pad=20)

# 🔹 הצגת התרשים
plt.show()

print("💡 הלקוחות שחשופים לקמפיינים מוציאים בכל רכישה משמעותית יותר מאנשים שלא חשופים לקמפיינים.")


# **🔹 יצירת DataFrame חדש עם הנתונים האמיתיים**
df_purchase_channels = df_grouped[[
    "CustomerCategory",
    "DiscountPurchasesPercentage",
    "WebPurchasesPercentage",
    "CatalogPurchasesPercentage",
    "StorePurchasesPercentage"
]].copy()

# **🔹 הגדלת הקנבס כדי לפנות מקום למקרא**
fig, ax = plt.subplots(figsize=(12, 8))

# **🔹 צבעים מודרניים לכל ערוץ**
colors = ["navy", "cornflowerblue", "lightsalmon", "darkred"]

# **🔹 יצירת תרשים עמודות מוערמות**
df_purchase_channels.set_index("CustomerCategory").plot(
    kind="bar", stacked=True, color=colors, width=0.8, ax=ax
)

# **🔹 עיצוב כותרות וצירים**
plt.xlabel("Customer Category", fontsize=14, fontweight='bold')
plt.ylabel("Purchase Distribution (%)", fontsize=14, fontweight='bold')
plt.title("Purchase Distribution by Channel", fontsize=16, fontweight='bold', pad=30)

plt.xticks(rotation=25, fontsize=12, fontweight='bold', ha='right')
plt.yticks(fontsize=12, fontweight='bold')

# **🔹 הורדת המספר 120 מציר ה-Y אבל השארת השטח עד 100%**
ax.set_yticks(np.arange(0, 101, 20))  # מסמן רק אחוזים עד 100
ax.set_ylim(0, 110)  # מאפשר מקום לתוויות מעל הברים

# **🔹 הזזת המקרא למיקום נוח מעל התרשים**
legend = plt.legend(
    title="Purchase Channel", fontsize=12, title_fontsize=13, 
    loc="upper center", bbox_to_anchor=(0.5, 1.08), ncol=4, frameon=False
)

# **🔹 הוספת תוויות של אחוזים בתוך הברים**
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=12, fontweight='bold', color="white")

# **🔹 הוספת גריד אופקי עדין לשיפור קריאות**
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.xaxis.grid(False)


# **🔹 הצגת התרשים**
plt.show()

print("הלקוחות שלא נחשפים לקמפיינים, רוכשים יותר בהנחות (כי הם מחפשים את הדילים), אלה שחשופים לקמפיינים נוטים באופן יחסי לרכוש הקטלוג יותר משאר הקבוצות .")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import display

# 🔹 התחברות למסד הנתונים
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# 🔹 טעינת נתונים מהמסד
query = """
    SELECT c.CustomerID, c.IncomeCategory, 
           p.MntWines, p.MntMeatProducts, p.MntFishProducts, 
           p.MntSweetProducts, p.MntGoldProducts
    FROM Customers c
    INNER JOIN Purchases p ON c.CustomerID = p.CustomerID
"""
df = pd.read_sql(query, engine)

# 🔹 חישוב סך ההוצאות לכל לקוח
df["TotalSpendingPerCustomer"] = df[
    ["MntWines", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProducts"]
].sum(axis=1)

# 🔹 יצירת טבלה מסכמת לפי קבוצת הכנסה
income_purchases = df.groupby("IncomeCategory").agg(
    CustomerCount=("CustomerID", "count"),
    TotalWineRevenue=("MntWines", "sum"),
    TotalMeatRevenue=("MntMeatProducts", "sum"),
    TotalFishRevenue=("MntFishProducts", "sum"),
    TotalSweetRevenue=("MntSweetProducts", "sum"),
    TotalGoldRevenue=("MntGoldProducts", "sum"),
    TotalRevenue=("TotalSpendingPerCustomer", "sum"),
    AvgSpendingPerCustomer=("TotalSpendingPerCustomer", "mean"),
    SpendingStdDev=("TotalSpendingPerCustomer", "std"),
    MinSpending=("TotalSpendingPerCustomer", "min"),
    MaxSpending=("TotalSpendingPerCustomer", "max")
).reset_index()

# 🔹 חישוב אחוזי רכישה לכל מוצר
for product in ['Wine', 'Meat', 'Fish', 'Sweet', 'Gold']:
    income_purchases[f'{product}Percentage'] = (
        income_purchases[f'Total{product}Revenue'] / income_purchases["TotalRevenue"] * 100
    ).round(2)

# 🔹 מיון לפי סך ההכנסות
income_purchases = income_purchases.sort_values(by="TotalRevenue", ascending=False)

# 📋 הצגת הטבלה
display(income_purchases)

# 📊 גרף 1 - סך ההכנסות לפי קטגוריית הכנסה
plt.figure(figsize=(8, 5))
sns.barplot(
    data=income_purchases,
    x="IncomeCategory",
    y="TotalRevenue",
    hue="IncomeCategory",
    palette="Blues_r",
    legend=False
)
plt.xlabel("קטגוריית הכנסה")
plt.ylabel("סך הכנסות")
plt.title("סך הכנסות לפי קבוצת הכנסה")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

print("💡 תובנה:\nקבוצות ההכנסה הגבוהה והגבוהה מאוד מניעות את רוב ההכנסות, עם ממוצע רכישה גבוה. בקבוצה הגבוהה מאוד ניתן לראות גם קנייה מופרזת אצל חלק מהלקוחות.\nהקבוצות הבינונית והנמוכה נמצאות הרבה מתחת - הן גם מונות פחות לקוחות וגם עם רמות הוצאה נמוכות.")

# 📊 גרף 2 - טווח הוצאות לפי קבוצת הכנסה
plt.figure(figsize=(12, 6))
for index, row in income_purchases.iterrows():
    plt.plot([row['MinSpending'], row['MaxSpending']], [index, index], marker='o', color='gray')
    plt.scatter(row['AvgSpendingPerCustomer'], index, color='red', zorder=3, label='Avg Spending' if index == 0 else "")
    plt.scatter(row['SpendingStdDev'], index, color='blue', zorder=3, label='Spending Std Dev' if index == 0 else "")

    middle_point = (row['MinSpending'] + row['MaxSpending']) / 2
    plt.text(middle_point, index, f"לקוחות: {row['CustomerCount']}", fontsize=10, verticalalignment='center',
             color='black', weight='bold', ha='center')
    plt.text(row['AvgSpendingPerCustomer'], index, f"{row['AvgSpendingPerCustomer']:.1f}", fontsize=10,
             verticalalignment='bottom', color='red', ha='right')
    plt.text(row['SpendingStdDev'], index, f"{row['SpendingStdDev']:.1f}", fontsize=10,
             verticalalignment='bottom', color='blue', ha='left')

plt.yticks(range(len(income_purchases)), income_purchases['IncomeCategory'])
plt.xlabel("טווח הוצאה ללקוח")
plt.title("טווח הוצאות, ממוצע וסטיית תקן לפי קבוצת הכנסה")
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

print("💡 תובנה:\nסטיית התקן דומה מאוד בקבוצות הגבוהות, מעידה על עקביות בהרגלי רכישה.\nבקבוצת ההכנסה הבינונית יש יכולת צריכה גבוהה יחסית אך עדיין מתחת לבכירות.\nקבוצת ההכנסה הנמוכה כוללת מעט לקוחות עם הוצאה נמוכה במיוחד.")

# 📊 גרף 3 - התפלגות רכישות לפי מוצרים
plt.figure(figsize=(10, 6))
ax = income_purchases.set_index('IncomeCategory')[
    ['WinePercentage', 'MeatPercentage', 'FishPercentage', 'SweetPercentage', 'GoldPercentage']
].plot(
    kind='bar',
    stacked=True,
    colormap='viridis',
    width=0.8
)

plt.xlabel("קטגוריית הכנסה")
plt.ylabel("אחוז מתוך סך ההוצאות")
plt.title("התפלגות רכישות לפי סוג מוצר בכל קבוצת הכנסה")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle="--", alpha=0.7)

# הוספת תוויות עם אחוזים
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', label_type='center', fontsize=10, color='white')

# מקרא בצד ימין
plt.legend(
    title="קטגוריית מוצר",
    labels=["יין", "בשר", "דגים", "מתוקים", "תכשיטים"],
    loc='upper center',
    bbox_to_anchor=(0.5, 1.12),
    ncol=5
)
plt.tight_layout()
plt.show()

print("💡 תובנה:\nלקוחות בעלי הכנסה נמוכה נוטים לרכוש יותר תכשיטים ומתוקים - אולי מתוך מטרה לצרוך חוויות מיידיות.\nלקוחות בעלי הכנסה גבוהה צורכים יותר יין ובשר – מוצרים שנחשבים יוקרתיים יותר.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import display  # הצגת טבלה

# 🔹 התחברות למסד הנתונים
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# 🔹 טעינת הנתונים בפעם אחת עם JOIN במקום קריאות נפרדות
query = """
    SELECT c.CustomerID, c.Age, 
           a.TotalCatalogPurchases, a.TotalWebPurchases, a.TotalStorePurchases, a.TotalDealsPurchases
    FROM Customers c
    INNER JOIN Activity a ON c.CustomerID = a.CustomerID
"""
df = pd.read_sql(query, engine)

# 🔹 יצירת קבוצות גיל בהתאם לנתונים
bins = [29, 39, 49, 59, 69, 79]
labels = ["29-39", "39-49", "49-59", "59-69", "69-79"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

# 🔹 חישוב רכישות כוללות
df["TotalPurchases"] = df[["TotalCatalogPurchases", "TotalWebPurchases", "TotalStorePurchases", "TotalDealsPurchases"]].sum(axis=1)

# 🔹 קביעת רמות רכישה
def classify_purchase_level(total):
    if total <= 10:
        return 'Low Purchases (<= 10)'
    elif 11 <= total <= 20:
        return 'Medium Purchases (11-20)'
    else:
        return 'High Purchases (> 20)'
        

df["PurchaseLevel"] = df["TotalPurchases"].apply(classify_purchase_level)

# 🔹 יצירת טבלת סיכום
catalog_stats = df.groupby(["AgeGroup", "PurchaseLevel"], observed=False).agg(
    NumberOfCustomers=("CustomerID", "count"),
    TotalCatalogPurchases=("TotalCatalogPurchases", "sum"),
    AvgCatalogPurchasesPerCustomer=("TotalCatalogPurchases", "mean"),
    StdDevCatalogPurchases=("TotalCatalogPurchases", "std")
).reset_index()

# 🔹 חישוב יחס רכישות קטלוג מכלל הרכישות
total_purchases = df.groupby("AgeGroup", observed=False)["TotalCatalogPurchases"].sum()
catalog_stats["CatalogPurchaseRatio"] = catalog_stats.apply(
    lambda row: row["TotalCatalogPurchases"] / total_purchases[row["AgeGroup"]] if total_purchases[row["AgeGroup"]] > 0 else 0, axis=1
)

# Pivot להצגת סכומים לכל PurchaseLevel בקבוצת גיל
pivot_data = catalog_stats.pivot(index="AgeGroup", columns="PurchaseLevel", values="TotalCatalogPurchases")[["High Purchases (> 20)", "Medium Purchases (11-20)", "Low Purchases (<= 10)"]]

# ציור Stacked Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
pivot_data.plot(kind="bar", stacked=True, ax=ax, color=["#2C6B7A", "#C57B57", "#E4C66C"])

# הוספת תוויות
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + height / 2,
                f"{int(height)}",
                ha='center', va='center',
                fontsize=10, fontweight="bold", color="white"
            )

ax.set_title("Purchase Level Category", fontsize=16, weight="bold")
ax.set_ylabel("Sum of TotalCatalogPurchases", fontsize=12)
ax.set_xlabel("AgeCategory", fontsize=12)
ax.legend(title="Purchase Level")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

import plotly.graph_objects as go

# סידור הצירים
x_order = ["High Purchases (> 20)", "Medium Purchases (11-20)", "Low Purchases (<= 10)"]
y_order = ["29-39", "39-49", "49-59", "59-69", "69-79"]

# בניית מטריצות לערכים
z_matrix = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="AvgCatalog").reindex(index=y_order, columns=x_order).values
custom_matrix = merged.pivot(index="AgeGroup", columns="PurchaseLevel")  # נשתמש בהמשך לטולטיפ

# טקסט שיופיע בתוך כל תא
text_matrix = np.round(z_matrix, 2).astype(str)

# יצירת customdata: שלוש שכבות – avg, ratio, std
avg = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="AvgCatalog").reindex(index=y_order, columns=x_order).values
ratio = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="CatalogPurchaseRatio").reindex(index=y_order, columns=x_order).values
std = merged.pivot(index="AgeGroup", columns="PurchaseLevel", values="StdDevCatalogPurchases").reindex(index=y_order, columns=x_order).values

# הכנסת כל שלוש השכבות ל-customdata בצורה תקנית
customdata = np.stack([avg, ratio, std], axis=-1)

# 🎯 יצירת הגרף
fig = go.Figure(data=go.Heatmap(
    z=z_matrix,
    x=x_order,
    y=y_order,
    text=text_matrix,
    texttemplate="%{text}",
    colorscale='BrBG',
    colorbar=dict(title="Avg Purchases"),
    customdata=customdata,
    hovertemplate=
        "<b>Age Group:</b> %{y}<br>" +
        "<b>Purchase Level:</b> %{x}<br>" +
        "<b>Avg Catalog Purchases:</b> %{customdata[0]:.2f}<br>" +
        "<b>Catalog Purchase Ratio:</b> %{customdata[1]:.2f}<br>" +
        "<b>Std Dev (Catalog):</b> %{customdata[2]:.2f}<extra></extra>"
))

fig.update_layout(
    title="Avg Catalog Purchases per Customer by Age & Purchase Level",
    xaxis_title="Purchase Level",
    yaxis_title="Age Group",
    width=950,
    height=600,
    font=dict(family="Arial", size=12)
)

fig.show()





In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import plotly.graph_objects as go  # Use Plotly for both charts

# ✅ Database connection
server = 'Matan_Eiluz'
database = 'My_Project_Marketing_Analyst'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# ✅ Fetch relevant data
query = """
SELECT 
    c.CustomerID,
    c.Income,
    c.Kidhome,
    c.Teenhome,
    a.TotalWebPurchases,
    a.TotalStorePurchases,
    a.TotalCatalogPurchases,
    a.TotalDealsPurchases,
    p.MntWines,
    p.MntFruits,
    p.MntMeatProducts,
    p.MntFishProducts,
    p.MntSweetProducts,
    p.MntGoldProducts
FROM Customers c
INNER JOIN Activity a ON c.CustomerID = a.CustomerID
INNER JOIN Purchases p ON c.CustomerID = p.CustomerID
"""
df = pd.read_sql(query, engine)

# ✅ Create HouseholdType
df["HouseholdType"] = np.where(df["Kidhome"] + df["Teenhome"] == 0, "No Kids",
                        np.where(df["Kidhome"] + df["Teenhome"] == 1, "One Child", "Multiple Children"))

# ✅ Calculate Spending and Spending Percentage
df["Spending"] = df[[
    "TotalWebPurchases", "TotalStorePurchases",
    "TotalCatalogPurchases", "TotalDealsPurchases"
]].sum(axis=1)

df["SpendingPercentage"] = df["Spending"] / df["Income"] * 100

# ✅ Create summary statistics by HouseholdType
household_stats = df.groupby("HouseholdType").agg(
    CustomerCount=("CustomerID", "count"),
    AvgIncome=("Income", "mean"),
    AvgSpendingPerCustomer=("Spending", "mean"),
    TotalRevenue=("Spending", "sum"),
    SpendingPercentagePerCustomer=("SpendingPercentage", "mean"),
    WebPurchasesPercentage=("TotalWebPurchases", "mean"),
    StorePurchasesPercentage=("TotalStorePurchases", "mean"),
    CatalogPurchasesPercentage=("TotalCatalogPurchases", "mean"),
    DealsPurchasesPercentage=("TotalDealsPurchases", "mean")
).reset_index()

# Calculate KPIs for tooltips
total_revenue = household_stats["TotalRevenue"].sum()
avg_spending_per_customer = household_stats["AvgSpendingPerCustomer"].mean()
avg_income = household_stats["AvgIncome"].mean()
customer_count_percentage = (household_stats["CustomerCount"] / household_stats["CustomerCount"].sum()) * 100
revenue_contribution = (household_stats["TotalRevenue"] / total_revenue) * 100

# 🔹 First Chart: Stacked Bar Chart (Purchase Type Distribution by Household Type) using Plotly
# Define purchase types and colors to match the image
purchase_types = ["CatalogPurchasesPercentage", "DealsPurchasesPercentage", "StorePurchasesPercentage", "WebPurchasesPercentage"]
colors = ["#2C6B7A", "#E4C66C", "#C57B57", "#D67A53"]  # Colors from the image

# Normalize to percentages
household_stats_percent = household_stats.set_index("HouseholdType")[purchase_types]
household_stats_percent = household_stats_percent.div(household_stats_percent.sum(axis=1), axis=0) * 100

# Reset index for Plotly
plotly_df = household_stats_percent.reset_index()

# Create the stacked bar chart with Plotly
fig = go.Figure()

for idx, (purchase_type, color) in enumerate(zip(purchase_types, colors)):
    fig.add_trace(go.Bar(
        y=plotly_df["HouseholdType"],
        x=plotly_df[purchase_type],
        name=purchase_type.replace("PurchasesPercentage", ""),
        orientation='h',
        marker_color=color,
        text=plotly_df[purchase_type].apply(lambda x: f"{x:.2f}%" if x > 4 else ""),
        textposition='inside',
        textfont=dict(size=12, color="white", family="Arial, bold"),
        hovertemplate=(
            "Avg Spending Per Customer: %{customdata[0]:.2f}<br>"
            "Avg Income: %{customdata[1]:.2f}<br>"
            "Total Revenue: %{customdata[2]:.2f}<br>"
            "Customer Count %: %{customdata[3]:.2f}%<br>"
            "Revenue Contribution: %{customdata[4]:.2f}%<extra></extra>"
        ),
        customdata=np.stack([
            [avg_spending_per_customer] * len(plotly_df),
            [avg_income] * len(plotly_df),
            [total_revenue] * len(plotly_df),
            customer_count_percentage,
            revenue_contribution
        ], axis=-1)
    ))

# Update layout to match the style
fig.update_layout(
    barmode='stack',
    title=dict(
        text="Purchase Distribution (%) by Household Type",
        font=dict(size=16, family="Arial, bold"),
        y=0.98,  # Higher title
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(
        tickformat="%",  # Keep percentage format for tooltips and bar labels
        range=[0, 100],
        tickfont=dict(size=12),
        showticklabels=False  # Hide the numerical tick labels on the x-axis
    ),
    yaxis=dict(
        tickfont=dict(size=12, family="Arial, bold")
    ),
    legend_title="Purchase Type",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1,  # Legend below the title but higher than default
        xanchor="center",
        x=0.5,
        font=dict(size=12),
        title_font=dict(size=13)
    ),
    width=800,
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=100)  # Extra space for title and legend
)

# Show the plot (inline in Jupyter)
fig.show()

# 🔹 Second Chart: Heatmap (Product Spending by Household Type) using Plotly
product_types = ["MntFruits", "MntFishProducts", "MntGoldProducts", "MntMeatProducts", "MntSweetProducts", "MntWines"]
household_product_spending = df.groupby("HouseholdType")[product_types].sum()

# Rename columns to match the heatmap labels in the first image
household_product_spending.columns = ["FRUIT", "FISH", "GOLD", "MEAT", "SWEET", "WINE"]

# Reset index for Plotly
heatmap_df = household_product_spending.reset_index()

# Create the heatmap with Plotly
fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_df.drop(columns="HouseholdType").values,
    x=heatmap_df.columns[1:],  # Product types
    y=heatmap_df["HouseholdType"],
    colorscale="Reds",
    showscale=False,  # No color bar, as in the original
    hoverongaps=False
))

# Update layout to match the style and increase size
fig_heatmap.update_layout(
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(tickfont=dict(size=12)),
    yaxis=dict(tickfont=dict(size=12)),
    width=800,  # Increased width to take up more space
    height=600,  # Increased height to take up more space
    plot_bgcolor='white',
    paper_bgcolor='white'
)

# Show the heatmap (inline in Jupyter)
fig_heatmap.show()

print("📌 Insight: Customers without kids spend the most on **wine and meat**, while customers with one or more children distribute their spending more broadly across different categories.\n")